# Final Report - DeepSniffing: writing molecules for odor words

**Semester group project for MAI642 - Deep Learning, M.Sc. in Artificial Intelligence, University of Cyprus, 2026.**

**Collaborators:** Emmanouil Oikonomou, Konstantin Krasovitskiy, Anastasia Kyriakou.

This notebook is the deliverable. It reads the committed evaluations under `output/`, draws every figure inline,
and runs no model: each cell opens a JSON or a CSV and plots it. With the repository cloned, everything below
works on a laptop with `matplotlib` and `pandas`. The run behind the numbers is named in `RUN.txt`.

## Executive summary

One question, in three steps: can a small transformer learn the language of molecules well enough to write one for a
set of odor words, and to name the odor words of a molecule it has never seen?

| model | what it is | generator loss (test) | classifier ROC-AUC (test) |
|---|---|---|---|
| the decoder | 6.32 M parameters, 6 layers, width 256 | **0.7124** (perplexity 2.04) | **0.811** |
| ridge on Morgan bits | the classic baseline | not applicable | 0.736 |

The request is read: telling the model a molecule's own odor tokens lowers its loss by **0.212 nats** per molecule
token on test and **0.298** on DREAM, on 90 and 97 percent of molecules. What it writes is valid in every case, since
SELFIES cannot produce an invalid molecule, and 70 to 86 percent of the distinct structures appear in neither
training set. Against a ridge model on fingerprints the transformer is worth seven to eleven ROC-AUC points, and
the margin does not shrink on the hardest set.

The honest framing is in section 9: what came from the earlier version of this project, what we changed, and what
this report does not claim.

## The storyline

Three questions, in the order the project asked them:

1. **Does it learn the language at all?** Pretraining on 3.4 million PubChem molecules, then fine-tuning on 6,344
   molecules that sources name, does the model write molecules that hold together, weigh what a smellable molecule
   weighs, and differ from each other?
2. **Does the request do anything?** A generator that ignores its odor tokens would look fine on a loss curve. The
   test is whether the same molecule becomes more probable when the model is told its own odor words than when it
   is told nothing, and than when it is told another molecule's words.
3. **Is it new?** A model that reproduces the training set would score well and be useless. `novel` counts strings;
   the Tanimoto comparison, when it fills, says how far the nearest training molecule is.

A fourth question, "does a written molecule smell of what was asked", is the one we cannot answer. Nobody has
smelled the output, and no number here pretends otherwise.

## Palette

Every figure uses the same colours:

- **the decoder** - navy `#1F3864`
- **ridge on Morgan bits** - slate `#8FAADC`
- **chance** - red `#C00000`
- **secondary bars** - blue `#4472C4`

## 1. How the numbers were produced (the honest meta notes)

**Hardware and environment.** The committed run was executed on a **Colab Pro L4**, from commit `e33cda6`, with
torch 2.14.1, rdkit 2026.9.1 and selfies 2.2.0, all recorded in `RUN.txt`. The environment comes from
`poetry.lock`, so a rerun gets the same versions. PyPI would serve whatever is newest that day.

**One run, not a study.** The pretraining ran 24 epochs, its best validation 0.7857 at epoch 23. The generator ran its
documented 160 epochs, the classifier one pass for its head. There is no seed sweep here: the earlier notes record
training noise of about 0.004, and we did not measure it for this run. Differences smaller than that should not be
read.

**The data changed under the project.** An earlier run reported 0.7062, 0.7096 and 0.6759. It was measured before
the fix that removed twelve DREAM molecules, which no odor source describes, from the pretraining: they were held
out by CID alone, and PubChem registers one structure under several CIDs. Those numbers are not comparable on
DREAM, and the earlier pipeline is preserved at the tag `archive/structure-to-odor-profile`.

**Generation is one sample.** The 200 molecules per request depend on the requests and the seed, not on the split,
which is why the same table appears once.

## 2. The data

Committed as `data/processed/odor/odor_dataset.csv`, 7,516 molecules and 890 odor tokens. The summary is written by
`python -m scripts.analysis.eda`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
eda = json.loads((ROOT / "output/analysis/eda.json").read_text())
splits = eda["splits"]
per = eda["tokens_per_molecule"]
pd.DataFrame(
    {
        "measure": ["molecules", "odor tokens", "tokens a molecule", "sources", "median weight", "over 350 Da"],
        "value": [
            f"{eda['molecules']:,}: {splits['train']:,} train, {splits['validation']} validation, {splits['test']} test, {splits['dream']} DREAM",
            f"{eda['odor_tokens']}, {eda['odor_tokens_on_one_molecule']} on a single molecule",
            f"median {per['median']:.0f}, mean {per['mean']:.1f}, most {per['most']}",
            f"{len(eda['sources'])}",
            f"{eda['weight']['median']:.0f} Da",
            f"{eda['weight']['share_over_350']:.1%}",
        ],
    }
).set_index("measure")

**Reading.** The vocabulary is wide and thin: `fruity` carries 2,713 molecules and 211 terms carry one. Each
molecule carries 5.8 tokens on average, and the generator is asked for one to six of them, so it never sees a
molecule's full description at once. The splits are by CID, with a fixed seed, and the DREAM molecules are held out
of everything including the pretraining; the check that reads the built table reports 0 of 480 by CID and 0 by
structure.

## 3. The model

A decoder-only transformer of six layers, eight heads and width 256, which is **6,319,616 parameters**, of which
531,712 are the token table and 786,432 the six request adapters. The vocabulary has 2,077 tokens: 6 special, 1,181
molecule tokens and 890 odor tokens, fixed before pretraining so the saved weights are tied to it.

The odor tokens of a request are a set, not a sequence: they have no position, they cannot see each other, and the
request adapter averages them into one vector that scales and shifts the state at every layer, from the separator
on. The weights of that adapter start at zero, so fine-tuning begins from exactly the pretrained model.

### The three loss curves

Each stage saves its history after every epoch: 24 epochs of pretraining, 160 of the generator, and the
classifier's single pass over the molecules.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 5.5))
stages = ("stage1", "stage2", "stage3")
names = ("pretraining", "generator", "classifier")
metrics = ("validation_accuracy", "request_gain", "roc_auc")
labels = ("share of tokens ranked first", "request gain (nats)", "ROC-AUC")
for column, (stage, name, metric, label) in enumerate(zip(stages, names, metrics, labels, strict=True)):
    history = pd.read_csv(ROOT / f"output/odor/{stage}/history.csv")
    axes[0, column].plot(history["epoch"], history["train"], color="#8FAADC", label="train")
    axes[0, column].plot(history["epoch"], history["validation"], color="#1F3864", label="validation")
    axes[0, column].set_title(name)
    axes[0, column].set_xlabel("epoch")
    axes[1, column].plot(history["epoch"], history[metric], color="#C00000")
    axes[1, column].set_ylabel(label)
    axes[1, column].set_xlabel("epoch")
axes[0, 0].set_ylabel("loss")
axes[0, 0].legend(frameon=False)
plt.tight_layout()
plt.show()

for stage in stages:
    history = pd.read_csv(ROOT / f"output/odor/{stage}/history.csv")
    best = history.loc[history["validation"].idxmin()]
    print(f"{stage}: best validation {best['validation']:.4f} at epoch {int(best['epoch'])}, last train {history['train'].iloc[-1]:.4f}")


## 4. The generator: how probable the real molecules are

Scored in nats per molecule token, with the request excluded from the loss. Lower is better; the gain is the loss
without a request minus the loss with the molecule's own tokens.

### What fine-tuning did to the odor tokens

The embeddings themselves hardly moved: a median relative change of 3.1 percent, a cosine of 0.9996 between the
pretrained and the fine-tuned vector, and not one of the 890 tokens below 0.99. What moved is the six request
adapters, from exactly zero to norms of 4.3, 4.8, 5.6, 6.4, 7.5 and 8.9, rising with depth as layer decay predicts.

The model did not relearn what the odor words mean; it learned how to use them. `analysis_representations.ipynb`
draws both maps and the neighbours that changed.

In [ ]:
SPLITS = ("validation", "test", "dream")
generator = {s: json.loads((ROOT / f"output/odor/evaluation_{s}.json").read_text()) for s in SPLITS}
rows = []
for split in SPLITS:
    found = generator[split]["conditioning"]
    gain = found["gain_over_no_odor_tokens"]
    rows.append(
        {
            "split": split,
            "molecules": found["molecules"],
            "loss": round(found["loss_with_own_odor_tokens"], 4),
            "perplexity": round(found["perplexity_with_own_odor_tokens"], 2),
            "gain": f"{gain['mean']:+.3f} +/- {gain['interval_95']:.3f}",
            "helped": f"{gain['helped']:.0%}",
            "vs a disjoint request": f"{found['gain_over_other_odor_tokens']['mean']:+.3f}",
        }
    )
pd.DataFrame(rows).set_index("split")

**Reading.** 0.7112 on validation, 0.7124 on test, 0.6972 on DREAM, a perplexity near 2 in each case. The
gain is +0.212 +/- 0.020 on test and +0.298 +/- 0.016 on DREAM, and 90 and 97 percent of molecules are helped.
DREAM is easier than test by 0.086 nats of gain, which is worth noticing: those molecules come from another
collection, and the model finds them easier to place once told their odor tokens.

Against a request built from a molecule that shares none of its tokens, the own tokens are worth +0.217, +0.196
and +0.207 nats on the three splits, and 91, 89 and 92 percent of molecules prefer their own. The model reads which
odor tokens were asked for, not merely that some were.

## 5. The classifier: naming the odor tokens of a molecule

Macro ROC-AUC over the tokens that can be scored, meaning those with at least one molecule that carries them and
one that does not. Chance is 0.5.

In [ ]:
classifier = {s: json.loads((ROOT / f"output/odor/stage3/evaluation_{s}.json").read_text()) for s in SPLITS}
baseline = json.loads((ROOT / "output/analysis/baseline.json").read_text())
pd.DataFrame(
    [
        {
            "split": split,
            "molecules": classifier[split]["molecules"],
            "tokens scored": classifier[split]["odor_tokens_scored"],
            "ROC-AUC": round(classifier[split]["roc_auc_macro"], 3),
            "average precision": round(classifier[split]["average_precision_macro"], 3),
            "tokens seen 10+": round(classifier[split]["roc_auc_macro_odor_tokens_seen_10_times"], 3),
            "precision at 5": round(classifier[split]["precision_at_5"], 3),
        }
        for split in SPLITS
    ]
).set_index("split")

**Reading.** 0.832, 0.811 and 0.776 across validation, test and DREAM, with average precision of 0.224, 0.268 and
0.224 and precision at 5 of 0.359, 0.381 and 0.580. The precision figure is the honest one for a search: of the five
odor tokens the model ranks highest for a molecule, between a third and a half are real. Restricted to the 44
tokens seen ten or more times, test rises to 0.830 and DREAM to 0.813, so the macro average is mostly dilution by
the rare ones.

## 6. Token by token

The macro average is one number over hundreds of tokens and hides both ends. This is the distribution behind it on
the test molecules.

In [ ]:
aucs = np.array([row["auc"] for row in classifier["test"]["per_token"]])
support = np.array([row["molecules"] for row in classifier["test"]["per_token"]])

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.hist(aucs, bins=40, color="#4472C4")
ax.axvline(np.median(aucs), color="#1F3864", linestyle="--", linewidth=1, label=f"median {np.median(aucs):.3f}")
ax.axvline(0.5, color="#C00000", linestyle="--", linewidth=1, label="chance")
ax.set_xlabel("ROC-AUC of one odor token")
ax.set_ylabel("tokens")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"{len(aucs)} of 890 tokens scored; median {np.median(aucs):.3f}; {np.mean(aucs < 0.5):.0%} worse than chance")
print(f"on 1-2 molecules: {(support <= 2).sum()} tokens, median {np.median(aucs[support <= 2]):.3f}")
print(f"on 10 or more:    {(support >= 10).sum()} tokens, median {np.median(aucs[support >= 10]):.3f}")

In [ ]:
worst = sorted(classifier["test"]["per_token"], key=lambda row: row["auc"])[:10]
pd.DataFrame(worst).assign(auc=lambda d: d["auc"].round(3)).set_index("token")

**Reading.** The median token is 0.877 and the average is 0.811, because 31 of the 305 tokens are ranked worse than chance and
29 of those sit on one or two molecules. Singletons are not worse on average, 0.909 against 0.849, but their tenth
percentile is 0.368 against 0.702: one positive molecule can land at the top by luck, or at the bottom. The
frequent tokens are not the easy ones: the 44 that appear on ten or more test molecules have a median of 0.849,
below the 0.909 of the singletons. The ten worst are `linden` at 0.043, `linden flower` 0.057, `powerful` 0.068,
`fat` 0.134, `concord grape` 0.154, `sea` 0.193, `opoponax` 0.197, `bean green bean` 0.199, `tart` 0.225 and
`peppery` 0.254. Every one of them is carried by one or two molecules.

## 7. Against the ridge baseline

Ridge on 2,048 Morgan bits, fit on the same 6,344 training molecules for the same 890 tokens, scored the same way.

In [ ]:
values = np.array([[baseline["splits"][s]["roc_auc_macro"], classifier[s]["roc_auc_macro"]] for s in SPLITS])
x = np.arange(len(SPLITS))
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(x - 0.19, values[:, 0], width=0.38, label="ridge on Morgan bits", color="#8FAADC")
ax.bar(x + 0.19, values[:, 1], width=0.38, label="the decoder", color="#1F3864")
for i, (ridge, neural) in enumerate(values):
    ax.text(i - 0.19, ridge + 0.01, f"{ridge:.3f}", ha="center", fontsize=8)
    ax.text(i + 0.19, neural + 0.01, f"{neural:.3f}", ha="center", fontsize=8)
ax.set_xticks(x, SPLITS)
ax.set_ylim(0.5, 0.95)
ax.set_ylabel("macro ROC-AUC")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

for split, (ridge, neural) in zip(SPLITS, values, strict=True):
    print(f"{split:10s} ridge {ridge:.3f}  decoder {neural:.3f}  difference {neural - ridge:+.3f}")

**Reading.** The decoder is worth +0.107 on validation, +0.075 on test and +0.106 on DREAM. The margin is
consistent, which matters more than its size: a single split could be luck. The baseline is not weak, either. Ridge
on fingerprints has been the standard for molecular property prediction for a decade, and at 0.736 on test it is
well above chance, so the transformer adds something a classic model lacks, and it is not merely doing a task only a network could do
at all. Average precision tells a narrower story: 0.224 against 0.223 and 0.268 against 0.261 on validation and test,
and 0.224 against 0.178 on DREAM. The ranking edge is consistent; the precision edge appears mainly on DREAM.

## 8. What it wrote

200 molecules for each of seven requests, six sets of odor tokens and one empty request.

In [ ]:
generation = generator["test"]["generation"]
pd.DataFrame(
    [
        {
            "request": request,
            "valid": round(found["valid"], 2),
            "unique": round(found["unique"], 2),
            "novel": round(found["novel"], 2),
            "median weight": round(found["weight_median"]),
            "over 350": f"{found['weight_over_350']:.1%}",
        }
        for request, found in generation.items()
    ]
).set_index("request")

The same numbers as bars, because the comparison across requests is what the table is for.

In [ ]:
labels = list(generation)
x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(x - 0.27, [generation[r]["valid"] for r in labels], 0.27, label="valid", color="#1F3864")
ax.bar(x, [generation[r]["unique"] for r in labels], 0.27, label="unique", color="#4472C4")
ax.bar(x + 0.27, [generation[r]["novel"] for r in labels], 0.27, label="novel", color="#8FAADC")
ax.set_xticks(x, labels, rotation=25, ha="right")
ax.set_ylim(0.6, 1.05)
ax.set_ylabel("share")
ax.legend(frameon=False, ncols=3)
plt.tight_layout()
plt.show()

**Reading.** Valid is 1.00 everywhere, which is what SELFIES buys: every string decodes to some molecule. Unique
runs from 0.89 to 1.00 and novel from 0.70 to 0.86, so the sampler is not repeating one structure in different
words. The empty request is the exception: 0.99 novel, median weight 271 Da, 6 percent over 350, the largest of the
seven. With nothing asked for the model falls back on the pretraining distribution, which is exactly what replay
teaches it to do.

Structure similarity does not confirm the request. For `fruity`, a written molecule sits 0.67 from the nearest
fruity training molecule and 0.68 from the nearest of any kind; for `musk`, 0.48 and 0.61, so it is further from
real musk molecules than from the corpus. The weights are right in both cases, which is what a conditioner that
has learned the size of the answer and not its neighbourhood looks like.

## 9. What we reproduced, what we added, and what we did not claim

The project changed direction mid-semester, and the record should say so.

**What came from the earlier version.** The DREAM Olfaction Prediction pipeline, a hybrid over descriptors and
Morgan fingerprints that scored 6.896 on the official Q2 metric, is at the tag
`archive/structure-to-odor-profile` with the material no commit holds on its release. It is the evidence that
structure carries enough information about odor to be worth modelling, and it is where the evaluation discipline
came from.

**What we changed.** The decoder is not an improvement of that model; it is a different question. The earlier work
predicted 21 ratings from a descriptor table, and was judged to read as machine learning rather than deep learning.
The decoder makes the learning problem the language of molecules: 3.4 million pretraining molecules, a request
injected at every layer, and a classifier on frozen representations. None of that can be replaced by a random
forest.

**What we added.** The leak fix and its check, the per-token analysis, the ridge baseline on the same split, the
reproducible run with its artifacts in the tree, and the analysis notebooks.

**What we did not claim.** We did not smell anything. We did not compare the generator against a published
baseline, because there is no comparable task with a public leaderboard. We did not measure training noise on this
run. We did not test a larger model, which is the obvious next lever. And the numbers here do not sit beside the earlier ones: they replace them,
because the data changed under both.

## 10. Verdict

**The generator.** It reads its request and writes molecules that hold together: 0.7124 test loss, +0.212 nats of
gain, 90 percent of molecules helped, valid in every case and mostly new. What it cannot tell us is whether the
molecules smell of what was asked, and that is the one question the field has no benchmark for.

**The classifier.** 0.811 macro ROC-AUC on test, and clearly better than ridge on fingerprints. It is not uniformly
good: it is good on the 44 tokens with enough molecules to learn and to measure, and close to useless on the long
tail of singletons behind the average. Precision at 5 of 0.38 is the number to quote for any practical use.

**Together.** The two halves agree about what the model knows: it has learned the shape of molecules and something
about how odor words attach to them, and it has not learned the rare vocabulary, which is a data problem more than a
modelling one.

## 11. Future work

- **Capacity.** Pretraining plateaued at 0.7857 by epoch 23 with 6.3 million parameters, and the run left 18 GB of
  the L4 unused because the batch is capped by a constant. A larger model and a larger batch are the two levers
  nobody has pulled yet.
- **A latent space.** A conditional VAE on top of the decoder would give a molecule a vector that can be edited and
  interpolated. The request adapters already inject a vector at every layer, so the mechanism exists.
- **The visualisations.** Where in the decoder the request acts, and how the odor token embeddings move between
  pretraining and fine-tuning.
- **A smell check.** The only way to know whether the output smells right is to have chemists smell it. That is a
  study, not a model.
- **The vocabulary tail.** Merging the 211 singleton tokens, or requiring a minimum support before training, would
  change what the classifier is asked to do and probably raise the headline number honestly.

## 12. References and how to run everything

The data comes from Pyrfume at a pinned revision, PubChem's CID-SMILES and CID-Mass archives, and the DREAM
Olfaction Prediction Challenge list. The model is a decoder-only transformer of the kind used for language, applied
to SELFIES strings.

```bash
poetry install
ds-prepare            # the odor sources
ds-build              # PubChem, the tables, the vocabulary, the packed tokens
ds-train --stage 1    # 24 epochs, about two hours on an L4
ds-train --stage 2    # 160 epochs, the generator
ds-train --stage 3    # the classifier head
python -m scripts.core.evaluate --split test
python -m scripts.core.classifier --split test
python -m scripts.core.report          # results.md and the figures
python -m scripts.analysis.eda         # the data summary
python -m scripts.analysis.baseline    # the ridge baseline
```

`RUN.txt` names the commit, the GPU and the versions of the committed run.

## Appendix: the numbers in one place

In [ ]:
inventory = []
for split in SPLITS:
    found = generator[split]["conditioning"]
    inventory.append(
        {
            "split": split,
            "generator loss": round(found["loss_with_own_odor_tokens"], 4),
            "gain": round(found["gain_over_no_odor_tokens"]["mean"], 3),
            "classifier ROC-AUC": round(classifier[split]["roc_auc_macro"], 3),
            "ridge ROC-AUC": round(baseline["splits"][split]["roc_auc_macro"], 3),
            "tokens scored": classifier[split]["odor_tokens_scored"],
        }
    )
pd.DataFrame(inventory).set_index("split")